# 3. Follow the evidence

**Question:** Which observations demonstrate runtime behavior, and which establish security risk?

An invocation response, a content-filter block, an Application Insights trace, a Defender alert, and an Agent 365 risk signal are separate evidence. The repository has **not verified end-to-end Agent 365 ingestion or agent-attributed jailbreak risk**. This notebook does not invent a Defender alert from a blocked response or assume historical alert backfill.

Prompt agents have Foundry-managed collection. Hosted agents use the native hosting SDK and Microsoft OpenTelemetry. Neither path is automatically routed through the standalone APIM model API.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/governance').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from IPython.display import Markdown, display
from src.governance.evidence import synthetic_estate, load_snapshot, markdown_table

# Optional normalized CLI export; source events retain their original event times.
SNAPSHOT_PATH = None
estate = load_snapshot(Path(SNAPSHOT_PATH)) if SNAPSHOT_PATH else synthetic_estate()
rows = [row.model_dump(mode='json') for row in estate.evidence if row.category in ('control', 'security', 'visibility')]
display(Markdown(markdown_table(rows)))
assert not any(row['status'] == 'observed' and row['evidence_kind'] == 'unverified' for row in rows)

## How this agent integrates with Agent 365

For **observability**, Microsoft's current integration is `microsoft-opentelemetry`. It replaces the earlier Agent 365 **Observability SDK**, not the entire Agent 365 SDK. This notebook focuses on telemetry, not Microsoft 365 tool access, notifications, or identity provisioning.

| Runtime | Telemetry owner | Application responsibility |
| --- | --- | --- |
| Managed prompt agent | Foundry service | Invoke the existing agent; account collection and service prerequisites are separate |
| Hosted agent | Native hosting SDK plus Microsoft OpenTelemetry | Use the host entry point; preserve platform identity context and privacy defaults |

The hosted flow is `ResponsesHostServer` → `azure-ai-agentserver-core` → Microsoft OpenTelemetry → configured telemetry destinations. Agent 365 and Application Insights are distinct destinations. An Application Insights connection string is **not** an Agent 365 credential.

The next cell displays installed package versions and the **actual source assignment** from the repository. It reads the file without importing or starting the hosted agent.

In [ ]:
from src.governance.agent365_walkthrough import integration_packages, hosted_entrypoint_source, inspect_native_wiring

display(Markdown(markdown_table(integration_packages())))
display(Markdown('### Repository hosting entry point\n```python\n' + hosted_entrypoint_source() + '\n```'))

## Execute the SDK configuration path without exporting

`inspect_native_wiring` calls the installed host's public `configure_observability` function. It **mocks `use_microsoft_opentelemetry` before that function runs**. The host therefore constructs its configuration, but the distro initializer, exporters, instrumentation, and token acquisition never execute.

The example simulates only hosted feature flags, once disabled and once enabled. It supplies **no fabricated agent identity or tenant ID**. An inert placeholder makes the Azure Monitor configuration branch visible; it is not read from `.env` or sent anywhere. Logger mutations are intercepted, environment changes are restored, and only selected booleans are displayed—not raw resource/identity configuration.

In production, Foundry supplies the hosted platform flags and identity context. **Do not copy the mock feature flags into a local agent to pretend it is hosted, or initialize a second telemetry provider alongside `ResponsesHostServer`.**

In [ ]:
wiring = [inspect_native_wiring(agent365_enabled=enabled) for enabled in (False, True)]
display(Markdown(markdown_table(wiring)))
assert [row['agent365_requested'] for row in wiring] == [False, True]
assert all(row['azure_monitor_requested'] for row in wiring)
assert all(not row['sensitive_content_enabled'] and not row['exporter_executed'] for row in wiring)
print('Verified configuration branching only. No telemetry sent; ingestion and risk remain unverified.')

## Identity, privacy, and prerequisites versus delivery

Foundry's agent instance, blueprint, tenant, and project identities are not interchangeable. The host enriches telemetry with platform context; a caller-controlled APIM `x-agent-name` header is not that identity. The offline table does not verify a real identity or permission assignment.

The host defaults to sensitive-content recording disabled. The repository's search tool logs only whether glossary expansion happened, not the raw question or expanded query. This matters because application logs can also be exported; disabling span content alone is not a general log-redaction policy.

The existing read-only configuration diagnostic has this CLI interface (it is **not run by this notebook**):

```text
python -m scripts.demo.agent365_preflight --hosted-agent-principal-id <agent-identity-object-id>
```

It checks collection state, the observability service and supplied hosted identities' permissions, and Defender configuration. It grants no permissions and sends no prompts. Exit `1` means a configuration/read failure; exit `2` means ingestion or coverage remains unknown. Optional snapshot rows below can carry results of real configuration reads, explicitly labeled as snapshots.

**Configuration evidence is not delivery evidence.** A fresh attributed session must be observed at the destination to establish ingestion. A separately attributed security signal must arrive to establish Agent 365 risk. Neither is proved by the mocked initializer or by a content-filter response.

References: [Microsoft OpenTelemetry Distro](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/microsoft-opentelemetry), [Foundry integration](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/agent-365-integration), and [hosted observability permissions](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/grant-agent-365-permissions).

## Optional synthetic input tests against existing endpoints

Two gates separate paid inference from the jailbreak test. Both are false by default. Normal calls use fictional HR questions; the jailbreak option requests hidden instructions as a synthetic guardrail test. It can produce security events. It does **not** mark an identity compromised or alter any policy.

The wrapper invokes existing prompt/hosted endpoints with no provisioning and no local-search fallback. Request metadata may provide correlation IDs, but a request ID is not necessarily an OpenTelemetry trace ID. Model output and prompt text are not exported.

In [ ]:
ALLOW_PAID_INFERENCE = False
ALLOW_JAILBREAK_TEST = False

if ALLOW_PAID_INFERENCE:
    from dotenv import load_dotenv
    from src.governance.live import invoke_agent, invoke_gateway
    load_dotenv(ROOT / '.env')
    results = [
        invoke_gateway(allow_inference=True, jailbreak=ALLOW_JAILBREAK_TEST),
        invoke_agent('HRPolicyAgent', allow_inference=True, jailbreak=ALLOW_JAILBREAK_TEST),
        invoke_agent('hr-policy-agent', hosted=True, allow_inference=True, jailbreak=ALLOW_JAILBREAK_TEST),
    ]
    display(Markdown(markdown_table([row.model_dump(mode='json') for row in results])))
else:
    print('Inference and jailbreak tests disabled. No prompts sent.')

## Attribution and drill-through

| Evidence | Establishes | Does not establish |
| --- | --- | --- |
| Agent definition inventory | Agent exists in a project | Publication, ingestion, or active risk |
| HTTP filter response | Filter outcome for that request | Defender detection or Agent 365 risk |
| Correlated trace | Observed execution at the trace source | A security finding |
| Configuration preflight | Collection/permission prerequisites | Export delivery or historical backfill |
| Account-matched Defender source alert | Detection at the source account | Specific agent attribution or Agent 365 arrival |

The live collector exposes selected account-level Defender alert metadata without suspicious-prompt evidence. Agent 365 risk feeds, Purview findings, Sentinel incidents and agent-identity correlation remain explicit visibility gaps. Azure Government applicability must be checked separately from this Commercial demo.

Implementation reference: [Agent365Observability.md](../../docs/Agent365Observability.md).